In [11]:
import nflreadpy as nfl

In [12]:
import pandas as pd

In [13]:
import numpy as np

In [14]:
pbp=nfl.load_pbp(seasons=True)

In [ ]:
pbp_pandas = pbp.to_pandas()

In [ ]:
pbp_pandas["score_diff"] = pbp_pandas["total_home_score"] - pbp_pandas["total_away_score"]

In [8]:
pbp_pandas["home_win"] = (pbp_pandas["result"] > 0).astype(int)

In [9]:
pbp_pandas["next_play"] = pbp_pandas["play_type"].shift(-1)

In [10]:
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

features = [
    "game_seconds_remaining",
    "down",
    "ydstogo",
    "yardline_100",
    "score_diff",
]

# scale only score_diff, pass everything else through w/o changes
preprocess = ColumnTransformer(
    [("score_diff_scaler", StandardScaler(), ["score_diff"])],
    remainder="passthrough",
)

model = Pipeline([
    ("preprocess", preprocess),
    ("clf", LogisticRegression(max_iter=1000)),
])

train = pbp_pandas.dropna(subset=features + ["home_win"])
X=train[features]
y=train["home_win"]
#model.fit(train[features], train["home_win"])
model.fit(X, y)
joblib.dump(model, "wp_model.joblib")

KeyError: ['score_diff']

In [22]:
def predict_wp_dictionary(model, game_state):

    features = [
        "game_seconds_remaining",
        "down",
        "ydstogo",
        "yardline_100",
        "score_diff"
    ]

    X = pd.DataFrame([game_state])[features]

    probabilities = model.predict_proba(X)[0]

    away_wp = probabilities[0]
    home_wp = probabilities[1]

    return [home_wp, away_wp]

In [23]:
gs1 = {
    "game_seconds_remaining": 600,
    "down": 2,
    "ydstogo": 6,
    "yardline_100": 45,
    "score_diff": 2
}


In [24]:
predict_wp_dictionary(model, gs1)

[np.float64(0.6072013732270544), np.float64(0.39279862677294564)]

In [26]:
def predict_wp_array(model, game_state):

    X = pd.DataFrame(
        [game_state],
        columns=[
            "game_seconds_remaining",
            "down",
            "ydstogo",
            "yardline_100",
            "score_diff",
        ],
    )

    probabilities = model.predict_proba(X)[0]

    # model classes are [0, 1] = [away, home]
    away_wp = probabilities[0]
    home_wp = probabilities[1]

    return [home_wp, away_wp]

In [27]:
gs1 = [600, 2, 6, 45, 2]

predict_wp_array(model, gs1)

[np.float64(0.6072013732270544), np.float64(0.39279862677294564)]

In [28]:
train = pbp_pandas[pbp_pandas["season"] <= 2024]
test  = pbp_pandas[pbp_pandas["season"] >= 2025]

X_train = train[features].dropna()
y_train = train.loc[X_train.index, "home_win"]

X_test = test[features].dropna()
y_test = test.loc[X_test.index, "home_win"]

preprocess = ColumnTransformer(
    [("score_diff_scaler", StandardScaler(), ["score_diff"])],
    remainder="passthrough",
)

model2 = Pipeline([
    ("preprocess", preprocess),
    ("clf", LogisticRegression(max_iter=1000)),
])

model2.fit(X_train, y_train)

test_probs = model2.predict_proba(X_test)[:, 1]   # P(home win)

brier = brier_score_loss(y_test, test_probs)
print("Brier score:", brier)

Brier score: 0.18027324462702563


In [1]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

features = [
    "game_seconds_remaining",
    "down",
    "ydstogo",
    "yardline_100",
    "score_diff",          # raw, not score_diff_z
]

train = pbp_pandas[pbp_pandas["season"] <= 2024]
test  = pbp_pandas[pbp_pandas["season"] >= 2025]

X_train = train[features].dropna()
y_train = train.loc[X_train.index, "home_win"]

X_test = test[features].dropna()
y_test = test.loc[X_test.index, "home_win"]

preprocess = ColumnTransformer(
    [("score_diff_scaler", StandardScaler(), ["score_diff"])],
    remainder="passthrough",
)

model2 = Pipeline([
    ("preprocess", preprocess),
    ("clf", LogisticRegression(max_iter=1000)),
])

model2.fit(X_train, y_train)

test_probs = model2.predict_proba(X_test)[:, 1]   # P(home win)

brier = brier_score_loss(y_test, test_probs)
print("Brier score:", brier)

NameError: name 'pbp_pandas' is not defined